In [3]:
!pip install -q datasets transformers accelerate

In [4]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import GPT2TokenizerFast



device = "cuda" if torch.cuda.is_available() else "cpu"


print('Pytorch version : ', torch.__version__)
print("Device : ", device)


if device == "cuda":
  print('GPU : ', torch.cuda.get_device_name(0))


#Downloading WIKItest- 2
dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1"
)


Pytorch version :  2.14.1+cu130
Device :  cuda
GPU :  Tesla T4


#Inpecting the dataset


In [5]:
print("Dataset splits : ")
print(dataset)


#inpect a few training examples
for i in range(10):
  print(f"\nExample  {i}:")
  print(repr(dataset["train"][i]["text"]))


#Count empty and non-empty records
train_texts = dataset["train"]["text"]


empty_count = sum(1 for text in train_texts if not text.strip())
non_empty_count = len(train_texts) - empty_count

print("\nDataset statistics")
print('Total records : ', len(train_texts))
print("Empty records : ", empty_count)
print("Non-Empty Counts : ", non_empty_count)

Dataset splits : 
DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})

Example  0:
''

Example  1:
' = Valkyria Chronicles III = \n'

Example  2:
''

Example  3:
' Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret blac

In [6]:
def prepare_text(split):

  lines = split["text"]

  cleaned_lines = [line.strip() for line in lines]

  corpus = "\n".join(cleaned_lines)

  return corpus


train_text = prepare_text(dataset['train'])
val_text = prepare_text(dataset["validation"])
test_text = prepare_text(dataset["test"])


#Inpect results
print('Training characters : ', len(train_text))
print("Validation characters : ", len(val_text))
print("Testing Characters : ", len(test_text))


print("\nPrepared training sample:")
print(train_text[:1000])

Training characters :  10858403
Validation characters :  1138523
Testing Characters :  1281306

Prepared training sample:

= Valkyria Chronicles III =

Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " .
The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard f

In [7]:
from transformers import GPT2TokenizerFast


#Load pretrained GPT-2 tokenizer
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")


#Set Padding token
tokenizer.pad_token = tokenizer.eos_token


#Tokenize training text
train_tokens = tokenizer(
    train_text,
    add_special_tokens = False
)


#inspect tokenization
print("Original text:")
print(train_text[:200])



print("\nToken IDs:")
print(train_tokens["input_ids"][:30])


#Checking how text is represented numerically.
print("\nDecoded tokens:")
print(tokenizer.decode(train_tokens["input_ids"][:30]))


#Measuring the number of tokens
print("\nTotal training tokens : ", len(train_tokens["input_ids"]))
print('Vocabulary size : ', tokenizer.vocab_size)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (2382565 > 1024). Running this sequence through the model will result in indexing errors


Original text:

= Valkyria Chronicles III =

Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Jap

Token IDs:
[198, 28, 569, 18354, 7496, 17740, 6711, 796, 198, 198, 10445, 73, 13090, 645, 569, 18354, 7496, 513, 1058, 791, 47398, 17740, 357, 4960, 1058, 10545, 230, 99, 161, 254]

Decoded tokens:

= Valkyria Chronicles III =

Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦�

Total training tokens :  2382565
Vocabulary size :  50257


#Verify Tokenization

In [8]:
val_tokens = tokenizer(
    val_text,
    add_special_tokens = False
)["input_ids"]


test_tokens = tokenizer(
    test_text,
    add_special_tokens = False
)["input_ids"]

print("Training tokens : ", len(train_tokens["input_ids"]))
print("Validation tokens : ", len(val_tokens))
print("Test Tokens : ",len(test_tokens))


print('Vocabulary size : ', tokenizer.vocab_size)

Training tokens :  2382565
Validation tokens :  246275
Test Tokens :  282071
Vocabulary size :  50257


#Creating Pytorch tensors

In [9]:
import torch

#Converting token ids into pytorch tensors
train_data = torch.tensor(
    train_tokens["input_ids"],
    dtype = torch.long
)

val_data = torch.tensor(
    val_tokens,
    dtype = torch.long
)

test_data = torch.tensor(
    test_tokens,
    #dtype=torch.long: Stores the token IDs as 64-bit integers.
    dtype = torch.long
)


#Inspecting tensor information
print("Training tensor : ", train_data.shape)
print("Validation tensor : ", val_data.shape)
print("Test Tensor : ", test_data.shape)


print("\nData Type: ", train_data.shape)
print("First 20 training token IDs : ", train_data[:20])

Training tensor :  torch.Size([2382565])
Validation tensor :  torch.Size([246275])
Test Tensor :  torch.Size([282071])

Data Type:  torch.Size([2382565])
First 20 training token IDs :  tensor([  198,    28,   569, 18354,  7496, 17740,  6711,   796,   198,   198,
        10445,    73, 13090,   645,   569, 18354,  7496,   513,  1058,   791])


In [10]:
#Create a batch function
block_size = 512
batch_size = 16


def get_batches(data):

  x = []
  y = []

  for i in range(batch_size):

    #Select a random starting position
    start = torch.randint(0, len(data) - block_size - 1, (1,)).item()


    #Get the next 512 tokens as input
    input_sequence = data[start: start + block_size]


    #Get the next 256 tokens as targets
    target_sequence = data[start + 1 : start + block_size + 1]


    #Store both sequences
    x.append(input_sequence)
    y.append(target_sequence)



  # Convert  lists into Pytorch tensors
  x = torch.stack(x)
  y = torch.stack(y)


  return x, y


#Generating one training batch
xb, yb = get_batches(train_data)

print("Input shape : ", xb.shape)
print("Target shape : ", yb.shape)


#Inspecting the sequences
print("Input sequences: ")
print(xb[0][:10])


print("\nTarget sequences: ")
print(yb[0][:10])











Input shape :  torch.Size([16, 512])
Target shape :  torch.Size([16, 512])
Input sequences: 
tensor([  345,   564,   247,  1569,   587, 21576,   564,   247,   764,   314])

Target sequences: 
tensor([  564,   247,  1569,   587, 21576,   564,   247,   764,   314,   564])


In [11]:
print("Input sequence:")
print(xb[0][:10])

print("\nTarget sequence:")
print(yb[0][:10])

print("\nAre they shifted correctly?")
print(torch.equal(xb[0][1:10], yb[0][:9]))

Input sequence:
tensor([  345,   564,   247,  1569,   587, 21576,   564,   247,   764,   314])

Target sequence:
tensor([  564,   247,  1569,   587, 21576,   564,   247,   764,   314,   564])

Are they shifted correctly?
True


#Token Embedding Layer

##There is a problem transformer doesn't work directly with token ids as meaningfull representations we need to convert each token id into a numerical vector

In [12]:
import torch.nn as nn

vocab_size = tokenizer.vocab_size
embedding_dim = 256

print('Vocabulary size : ', vocab_size)
print("Embedding Dimensions : ", embedding_dim)

Vocabulary size :  50257
Embedding Dimensions :  256


In [13]:
#Creating a Embedding layer


token_embedding = nn.Embedding(
    vocab_size,
    embedding_dim
)

In [14]:
#Testing the Embedding Layer
sample_embeddings = token_embedding(xb)


print('Input Shape : ', xb.shape)
print("Embedding Shape : ", sample_embeddings.shape)

Input Shape :  torch.Size([16, 512])
Embedding Shape :  torch.Size([16, 512, 256])


#Postional Embeddings

In [15]:
block_size = 512
embedding_dim = 256

positional_embedding = nn.Embedding(
    block_size,
    embedding_dim
)

#Generating Positions IDs
positions = torch.arange(block_size)
print(positions)

tensor([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,
         14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,  26,  27,
         28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,  39,  40,  41,
         42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,  53,  54,  55,
         56,  57,  58,  59,  60,  61,  62,  63,  64,  65,  66,  67,  68,  69,
         70,  71,  72,  73,  74,  75,  76,  77,  78,  79,  80,  81,  82,  83,
         84,  85,  86,  87,  88,  89,  90,  91,  92,  93,  94,  95,  96,  97,
         98,  99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111,
        112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125,
        126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139,
        140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153,
        154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167,
        168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 1

In [16]:
#Converting positions into embedding
positional_embeddings = positional_embedding(positions)

print('Position IDs Shape : ', positions.shape)
print('Position embedding shape : ', positional_embeddings.shape)

Position IDs Shape :  torch.Size([512])
Position embedding shape :  torch.Size([512, 256])


In [17]:
#Adding token and positional embedding

x = sample_embeddings + positional_embeddings

print('Token embedding shape : ', sample_embeddings.shape)
print("Positional Embedding shape : ", positional_embeddings.shape)
print('Final Embedding Shape : ', x.shape)

Token embedding shape :  torch.Size([16, 512, 256])
Positional Embedding shape :  torch.Size([512, 256])
Final Embedding Shape :  torch.Size([16, 512, 256])


#Self Attention

##Self-attention allows each token to look at other tokens and gather useful information from them.

In [18]:
import torch.nn as nn

#Creating Query, Key, Value layers

query_layer = nn.Linear(embedding_dim, embedding_dim)
key_layer = nn.Linear(embedding_dim, embedding_dim)
value_layer = nn.Linear(embedding_dim, embedding_dim)


#Generate Query , Key and Value
Q = query_layer(x)
K = key_layer(x)
V = value_layer(x)


#check Shapes
print('Query  shape : ', Q.shape)
print("Key shape : ", K.shape)
print("Value Shape : ", V.shape)

Query  shape :  torch.Size([16, 512, 256])
Key shape :  torch.Size([16, 512, 256])
Value Shape :  torch.Size([16, 512, 256])


#Calculation of attention score


In [19]:
attention_scores = Q @ K.transpose(-2,-1)

#Checking the shape
print('Attention Score shape : ', attention_scores.shape)

Attention Score shape :  torch.Size([16, 512, 512])


#Scaling
large dot-product values can make Softmax produce extremely concentrated probabilities

In [20]:
import math

#Get the dimension of key
head_dim = K.shape[-1]

#Scale attention Scores
scaled_attention_scores = attention_scores / math.sqrt(head_dim)

print('head Dimension : ', head_dim)
print('Scaled Attention scores shape : ', scaled_attention_scores.shape)

head Dimension :  256
Scaled Attention scores shape :  torch.Size([16, 512, 512])


#Applying Causal Masking

#currently our token attending the future token, we dont want out token to see the future token else it will learn and give the result but we the token to predict the next token

What does torch.tril() do?

It creates a lower triangular matrix:
- 1 means attention is allowed.
- 0 means attention is blocked.

In [21]:
#applying causal mask to scaled attention scores

masked_attention_scores = scaled_attention_scores.masked_fill(
    torch.tril(torch.ones_like(scaled_attention_scores)) == 0,
    float('-inf')
)

print('Masked Attention Scores shape : ', masked_attention_scores.shape)


#Display a samll portion
print(masked_attention_scores[0, :5, :5])

Masked Attention Scores shape :  torch.Size([16, 512, 512])
tensor([[ 1.0641,    -inf,    -inf,    -inf,    -inf],
        [-0.2893, -0.2727,    -inf,    -inf,    -inf],
        [-0.7703,  0.8898, -0.5636,    -inf,    -inf],
        [-0.3543,  0.2391, -0.5638,  0.4097,    -inf],
        [ 0.5561,  0.2058, -0.3883,  0.1349, -0.4588]],
       grad_fn=<SliceBackward0>)


#Applying softmax
we need to convert attention scores into probabilities (attention weights).

Softmax converts scores into values between 0 and 1, where the values across each row sum to 1.



In [22]:
#Create Softmax Layer
softmax = nn.Softmax(dim=-1)

#Convert scores into attention weights
attention_weights = softmax(masked_attention_scores)


#Check the output
print('Attention weights shape : ', attention_weights.shape)

#Check the sum of the attention weight for one token
print("Sum of attention weights : ", attention_weights[0,0].sum().item())


# Inspect the first 5 x 5 attention weights
print(attention_weights[0, :5, :5])

Attention weights shape :  torch.Size([16, 512, 512])
Sum of attention weights :  1.0
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4958, 0.5042, 0.0000, 0.0000, 0.0000],
        [0.1335, 0.7023, 0.1642, 0.0000, 0.0000],
        [0.1734, 0.3138, 0.1406, 0.3722, 0.0000],
        [0.3213, 0.2264, 0.1250, 0.2109, 0.1165]], grad_fn=<SliceBackward0>)


#Multiply attention weights with Value (V)

We have calculated how much attention each token should give to other tokens.

But we still need to use those attention weights to collect information from the Value vectors


Output = Attention Weights * V


In [23]:
#Multiply attention weights with Value Vectors
attention_output = attention_weights @ V

#Check Output Shape
print('Attention output Shape : ', attention_output.shape)

Attention output Shape :  torch.Size([16, 512, 256])


#Output Projection

The output projection is a learnable Linear layer applied after the attention operation. It transforms the combined information from the Value vectors into a representation that can be passed to the next Transformer component.

In [24]:
#Create output projection layer
output_projection = nn.Linear(embedding_dim, embedding_dim)


#Transform attention output
projected_output = output_projection(attention_output)


#Check Shape
print('Projected Output Shape : ', projected_output.shape)

Projected Output Shape :  torch.Size([16, 512, 256])


#Residual Connection


Output = x + projected_output


our original input embeddings contain:
- Token information.
- Positional information.
After self-attention and output projection, we have a transformed representation.

We don't want to lose the original input information. So we add the original input to the projected attention output.

In [25]:
#Applying residual connection
residual_output = x + projected_output

#Check Shapes
print("Original input Shape : ", x.shape)
print("Projected output shape : ", projected_output.shape)
print("Residual Output Shape : ", residual_output.shape)

Original input Shape :  torch.Size([16, 512, 256])
Projected output shape :  torch.Size([16, 512, 256])
Residual Output Shape :  torch.Size([16, 512, 256])


We are adding two tensors element by element:
- x: Original token and positional embeddings.
- projected_output: Information processed by self-attention.

##Residual connections help preserve information and improve gradient flow during training.

#Layer Normalization

After adding the original input and attention output, the resulting values can have different scales.

Layer Normalization helps keep the values in a more stable range, making training easier.

It normalizes the values across the embedding dimension for each token

In [26]:
#Creatinfg Layer Normalization layer
layer_norm = nn.LayerNorm(embedding_dim)

#Applying Layer Normalization
normalized_output = layer_norm(residual_output)

#Check Status
print('Residual output Shape : ', residual_output.shape)
print("Normalized Output Shape : ", normalized_output.shape)

Residual output Shape :  torch.Size([16, 512, 256])
Normalized Output Shape :  torch.Size([16, 512, 256])


#Feed Forward Network

self-attention has allowed tokens to gather information from other tokens.

But we also need a neural network to further process the information gathered by each token.

That's the job of the Feed-Forward Network (FFN).

In [27]:
#Creating the first linear layer
ffn_layer1 = nn.Linear(embedding_dim, embedding_dim*4)

#Applying the layer
ffn_output1 = ffn_layer1(normalized_output)

#Check Shape
print("Input Shape : ", normalized_output.shape)
print("FFN First Layer Output Shape : ", ffn_output1.shape)

Input Shape :  torch.Size([16, 512, 256])
FFN First Layer Output Shape :  torch.Size([16, 512, 1024])


#Applying ReLU Function
we'll apply an activation function to introduce non-linearity.

Negative values become zero.

Positive values remain unchanged.


In [28]:
#Creating ReLU activation
relu = nn.ReLU()

#Applying Relu
ffn_output2 = relu(ffn_output1)


#Check Shape
print("Before ReLU : ", ffn_output1.shape)
print('After ReLU : ', ffn_output2.shape)

Before ReLU :  torch.Size([16, 512, 1024])
After ReLU :  torch.Size([16, 512, 1024])


#Second Feed Forward Layer

Our first layer expanded each token representation from 256 to 1024 features. Now we bring it back to 256 so it matches the original embedding dimension.

In [29]:
ffn_layer2 = nn.Linear(embedding_dim * 4, embedding_dim)

#Applying the layer
ffn_output3 = ffn_layer2(ffn_output2)

#Check Status
print("Before second layer : ", ffn_output2.shape)
print("After Second Layer : ", ffn_output3.shape)

Before second layer :  torch.Size([16, 512, 1024])
After Second Layer :  torch.Size([16, 512, 256])


#FFN Residual Connection

In [30]:
#FFN residual Connection
ffn_residual_output = normalized_output + ffn_output3

#Check Shape
print("FFN Residual Output Shape : ", ffn_residual_output.shape)

FFN Residual Output Shape :  torch.Size([16, 512, 256])


#Second Layer Normalization

It helps keep the feature values well-scaled and makes training more stable.

In [31]:
#Creating the second Layer Normalization

layer_norm2 = nn.LayerNorm(embedding_dim)


#Applying Layer Normalization
final_output = layer_norm2(ffn_residual_output)

#Check shape
print('Before Layer Normalization : ', ffn_residual_output.shape)
print('After Layer Normalization : ', final_output.shape)

Before Layer Normalization :  torch.Size([16, 512, 256])
After Layer Normalization :  torch.Size([16, 512, 256])


Components completed
1. Query, Key and Value (Q, K, V) projections.
2. Attention score calculation using \(QK^T\).
3. Scaled dot-product attention.
4. Causal masking using torch.tril().
5. Applying -inf to future-token positions.
6. Softmax to calculate attention weights.
7. Attention output using attention_weights @ V.
8. Output projection.
9. Residual connection.
10. Layer Normalization.
11. Feed-Forward Network (256 → 1024 → 256).
12. ReLU activation.
13. FFN residual connection.
14. Second Layer Normalization.

#Next topic: Multi-Head Self-Attention

#Previously, we used one attention head with all 256 embedding dimensions.

#Now we'll divide those 256 dimensions across 4 independent attention heads.

#head_dim = embedding_dim\num_heads = 256/4=64

In [32]:
#Defining number of attention heads
num_heads = 4


#Calculate dimensions per head
head_dim = embedding_dim // num_heads


print("Embedding Dimensions : ", embedding_dim)
print("Number Of Heads : ", num_heads)
print("Dimensions Per Head : ", head_dim)

Embedding Dimensions :  256
Number Of Heads :  4
Dimensions Per Head :  64


#Create Query, Key and Value layers for Multi-Head Attention

In [33]:
#Creating Query, Key,and Value Projections layers
query_layer = nn.Linear(embedding_dim, embedding_dim)
key_layer = nn.Linear(embedding_dim, embedding_dim)
value_layer = nn.Linear(embedding_dim, embedding_dim)


#Applying projections to input embeddings
Q = query_layer(x)
K = key_layer(x)
V = value_layer(x)


#Check Shape
print("Query Shape : ", Q.shape)
print('Key Shape : ', K.shape)
print("V Shape : ", V.shape)

Query Shape :  torch.Size([16, 512, 256])
Key Shape :  torch.Size([16, 512, 256])
V Shape :  torch.Size([16, 512, 256])


#Reshape Query, Key, Value into 4 heads

In [34]:
#Reshape Query, Key and Value into 4 heads

Q = Q.view(batch_size, block_size,num_heads, head_dim)
K = K.view(batch_size, block_size, num_heads, head_dim)
V = V.view(batch_size, block_size, num_heads, head_dim)


#Check Shape
print("Query Shape : ", Q.shape)
print("Key Shape : ", K.shape)
print("Value Shape : ", V.shape)

Query Shape :  torch.Size([16, 512, 4, 64])
Key Shape :  torch.Size([16, 512, 4, 64])
Value Shape :  torch.Size([16, 512, 4, 64])


Now we need to arrange the dimensions so that PyTorch can calculate attention independently for each head.

Currently, our shape is:
[16, 512, 4, 64]


We want:
[16, 4, 512, 64]


In [35]:
#Transpose Query, Key and Value

Q = Q.transpose(1,2)
K = K.transpose(1,2)
V = V.transpose(1,2)

#Checking Shapes
print("Query Shape : ", Q.shape)
print("Key Shape : ", K.shape)
print("Value Shape : ", V.shape)

Query Shape :  torch.Size([16, 4, 512, 64])
Key Shape :  torch.Size([16, 4, 512, 64])
Value Shape :  torch.Size([16, 4, 512, 64])


#Calculating Attention Score

In [36]:
#Calculating Attention Score
attention_scores = Q @ K.transpose(-2,-1)

#Checking Shape
print("Query Shape : ", Q.shape)
print("Transposed Key Shape : ", K.transpose(-2,-1).shape)
print("Attention Scores Shape : ", attention_scores.shape)

Query Shape :  torch.Size([16, 4, 512, 64])
Transposed Key Shape :  torch.Size([16, 4, 64, 512])
Attention Scores Shape :  torch.Size([16, 4, 512, 512])


#Scaling Attention Score

In [37]:
import math

#Scaling Attention Scores
scaled_attention_scores = attention_scores/ math.sqrt(head_dim)

#Checking Shape
print("Head Dimension : ", head_dim)
print("Scaling Factor : ", math.sqrt(head_dim))
print("Scaled Attention shape : ", scaled_attention_scores.shape)

Head Dimension :  64
Scaling Factor :  8.0
Scaled Attention shape :  torch.Size([16, 4, 512, 512])


In [38]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("Available GPU memory:",
          round(torch.cuda.mem_get_info()[0] / 1024**3, 2), "GB")

Device: cuda
GPU: Tesla T4
Available GPU memory: 14.46 GB


#Applying Causal Masking


In [39]:
#Creating Causal Masking
causal_mask = torch.tril(torch.ones(block_size, block_size))

#Applying causal mask to attention scores
masked_attention_scores = scaled_attention_scores.masked_fill(
    causal_mask == 0,
    float("-inf")
)


#Check Shape
print('Causal Mask Shape : ', causal_mask.shape)
print("Masked Attention Shape : ", masked_attention_scores.shape)

Causal Mask Shape :  torch.Size([512, 512])
Masked Attention Shape :  torch.Size([16, 4, 512, 512])


#Applying softmax


In [40]:
#Creating Softmax Layer
softmax = nn.Softmax(dim = -1)

#Applying softmax
attention_weights = softmax(masked_attention_scores)

#Checking shapes
print("Attention Weights Shapes : ", attention_weights.shape)


#Checking Probability sum for one batch one hand, one token
print("First Row probability Sum : ", attention_weights[0,0,0].sum())

Attention Weights Shapes :  torch.Size([16, 4, 512, 512])
First Row probability Sum :  tensor(1., grad_fn=<SumBackward0>)


#Value Aggregation

In [41]:
#Calculating attention output
attention_output = attention_weights @ V


#Checking Shape
print('Attention Weights Shape : ', attention_weights.shape)
print("Value Shape : ", V.shape)
print("Attention Output Shape : ", attention_output.shape)

Attention Weights Shape :  torch.Size([16, 4, 512, 512])
Value Shape :  torch.Size([16, 4, 512, 64])
Attention Output Shape :  torch.Size([16, 4, 512, 64])


#Combining all heads


In [42]:
#Transpose the attention output
attention_output = attention_output.transpose(1,2)


#Combining all attention heads
attention_output = attention_output.contiguous().view(
    batch_size,
    block_size,
    embedding_dim
)


#Checking Shape
print("Combined Attention Output Shape : ", attention_output.shape)

Combined Attention Output Shape :  torch.Size([16, 512, 256])


We're combining the heads by concatenating their features, not adding their outputs together.

#Output Projection

we apply a learnable linear transformation to the combined attention output.

In [43]:
#Creating a output projection layer
output_projection = nn.Linear(embedding_dim, embedding_dim)

#Applying output projection
projected_output = output_projection(attention_output)


#Checking Shape
print("Before Projection : ", attention_output.shape)
print('After Projection : ', projected_output.shape)

Before Projection :  torch.Size([16, 512, 256])
After Projection :  torch.Size([16, 512, 256])


#Residual Connection

Now we'll add the original input x to our projected attention output.

In [44]:
#Applying residual connection
residual_output = x + projected_output

#Checking Shape
print("Original Input Shape : ", x.shape)
print("Projected Output Shape : ", projected_output.shape)
print("Residual Output Shape : ", residual_output.shape)

Original Input Shape :  torch.Size([16, 512, 256])
Projected Output Shape :  torch.Size([16, 512, 256])
Residual Output Shape :  torch.Size([16, 512, 256])


#Layer Normalization
After the residual connection, we apply LayerNorm to normalize the feature values


In [45]:
#Creating Layer Normalization
layer_norm = nn.LayerNorm(embedding_dim)

#Applying Layer Normalization
normalized_output - layer_norm(residual_output)


#Checking Shape
print("Before LayerNorm : ", residual_output.shape)
print("After LayerNorm : ", normalized_output.shape)

Before LayerNorm :  torch.Size([16, 512, 256])
After LayerNorm :  torch.Size([16, 512, 256])


#First FFN Layer


In [46]:
#Creating first FFN Layer
ffn_layer1 = nn.Linear(embedding_dim, embedding_dim * 4)


#Applying first FFN Layer
ffn_output1 = ffn_layer1(normalized_output)


#Checking Status
print("Input Shape : ", normalized_output.shape)
print("FFN First Layer Output Shape : ", ffn_output1.shape)

Input Shape :  torch.Size([16, 512, 256])
FFN First Layer Output Shape :  torch.Size([16, 512, 1024])


#Applying ReLU


In [47]:
#Creating ReLU activation
relu = nn.ReLU()


#Applying ReLU
ffn_output2 = relu(ffn_output1)


#Checking Shapes
print("Before ReLU : ", ffn_output1.shape)
print("After ReLU : ", ffn_output2.shape)



Before ReLU :  torch.Size([16, 512, 1024])
After ReLU :  torch.Size([16, 512, 1024])


#Second FFN Layer

In [48]:
#Creating second FFN Layer
ffn_layer2 = nn.Linear(embedding_dim * 4, embedding_dim)


#Applying Second FFN Layer
ffn_output3 = ffn_layer2(ffn_output2)


#Checking Shapes
print("Before Second Layer : ", ffn_output2.shape)
print("After Second Layer : ", ffn_output3.shape)

Before Second Layer :  torch.Size([16, 512, 1024])
After Second Layer :  torch.Size([16, 512, 256])


#FFN Residual Connection

In [49]:
#Applying FFN residual connection
ffn_residual_output = normalized_output + ffn_output3


#Checking Shape
print("FFN Residual Input Shape : ", normalized_output.shape)
print("FFN Output Shape : ", ffn_output3.shape)
print("FFN Residual Output Shape : ", ffn_residual_output.shape)

FFN Residual Input Shape :  torch.Size([16, 512, 256])
FFN Output Shape :  torch.Size([16, 512, 256])
FFN Residual Output Shape :  torch.Size([16, 512, 256])


#Second Layer Normalization

In [50]:
#Creating Second Layer Normalization
layer_norm2 = nn.LayerNorm(embedding_dim)


#Applying Layer Normalization
final_output = layer_norm2(ffn_residual_output)


#Checking Shapes
print("Before LayerNorm : ", ffn_residual_output.shape)
print("After LayerNorm : ", final_output.shape)

Before LayerNorm :  torch.Size([16, 512, 256])
After LayerNorm :  torch.Size([16, 512, 256])


#Creating the Multi-Head Attention class

In [51]:
import torch
import torch.nn as nn
import math

class MultiHeadSelfAttention(nn.Module):

    def __init__(self, embedding_dim, num_heads):

        super().__init__()

        # Store configuration
        self.embedding_dim = embedding_dim
        self.num_heads = num_heads
        self.head_dim = embedding_dim // num_heads

        # Query, Key and Value layers
        self.query_layer = nn.Linear(embedding_dim, embedding_dim)
        self.key_layer = nn.Linear(embedding_dim, embedding_dim)
        self.value_layer = nn.Linear(embedding_dim, embedding_dim)

        # Output projection
        self.output_projection = nn.Linear(embedding_dim, embedding_dim)

    def forward(self, x):

        batch_size, seq_length, embedding_dim = x.shape

        # Step 1: Generate Q, K and V
        Q = self.query_layer(x)
        K = self.key_layer(x)
        V = self.value_layer(x)

        # Step 2: Split into multiple heads
        Q = Q.view(batch_size, seq_length, self.num_heads, self.head_dim)
        K = K.view(batch_size, seq_length, self.num_heads, self.head_dim)
        V = V.view(batch_size, seq_length, self.num_heads, self.head_dim)

        # Step 3: Transpose dimensions
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        # Step 4: Calculate attention scores
        attention_scores = Q @ K.transpose(-2, -1)

        # Step 5: Scale attention scores
        attention_scores = attention_scores / math.sqrt(self.head_dim)

        # Step 6: Apply causal mask
        causal_mask = torch.tril(
            torch.ones(seq_length, seq_length, device=x.device)
        )

        masked_scores = attention_scores.masked_fill(
            causal_mask == 0,
            float("-inf")
        )

        # Step 7: Apply Softmax
        attention_weights = torch.softmax(masked_scores, dim=-1)

        # Step 8: Aggregate values
        attention_output = attention_weights @ V

        # Step 9: Combine all attention heads
        attention_output = attention_output.transpose(1, 2).contiguous()

        attention_output = attention_output.view(
            batch_size,
            seq_length,
            self.embedding_dim
        )

        # Step 10: Output projection
        output = self.output_projection(attention_output)

        return output

#Creating a test attention layer

In [52]:
multi_head_attention = MultiHeadSelfAttention(
    embedding_dim = embedding_dim,
    num_heads = num_heads
)


attention_test_output = multi_head_attention(x)

print("Input Shape : ", x.shape)
print("Attention Output Shape : ", attention_test_output.shape)

Input Shape :  torch.Size([16, 512, 256])
Attention Output Shape :  torch.Size([16, 512, 256])


#Building The Feed Forward Network


In [53]:
class FeedForwardNetwork(nn.Module):

    def __init__(self, embedding_dim):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(embedding_dim, embedding_dim * 4),

            nn.ReLU(),

            nn.Linear(embedding_dim * 4, embedding_dim)

        )

    def forward(self, x):

        return self.network(x)

#Testing the FFN

In [54]:
ffn = FeedForwardNetwork(
    embedding_dim = embedding_dim
)


ffn_test_output = ffn(x)

print("Input Shape : ", attention_test_output.shape)
print("FFN Output Shape : ", ffn_test_output.shape)

Input Shape :  torch.Size([16, 512, 256])
FFN Output Shape :  torch.Size([16, 512, 256])


A decoder block contains both attention and FFN. Later, we'll stack multiple decoder blocks to build our complete Transformer.

In [55]:
class DecoderBlock(nn.Module):

    def __init__(self, embedding_dim, num_heads):

        super().__init__()

        # Multi-Head Self Attention
        self.attention = MultiHeadSelfAttention(
            embedding_dim,
            num_heads
        )

        # First Layer Normalization
        self.layer_norm1 = nn.LayerNorm(embedding_dim)

        # Feed Forward Network
        self.ffn = FeedForwardNetwork(embedding_dim)

        # Second Layer Normalization
        self.layer_norm2 = nn.LayerNorm(embedding_dim)

    def forward(self, x):

        # Multi-Head Attention
        attention_output = self.attention(x)

        # Residual Connection + Layer Normalization
        x = self.layer_norm1(x + attention_output)

        # Feed Forward Network
        ffn_output = self.ffn(x)

        # Residual Connection + Layer Normalization
        x = self.layer_norm2(x + ffn_output)

        return x

#Creating a Testing Decoder Block

In [56]:
decoder_block = DecoderBlock(
    embedding_dim=embedding_dim,
    num_heads=num_heads
)

decoder_output = decoder_block(x)

print("Input Shape:", x.shape)
print("Decoder Output Shape:", decoder_output.shape)

Input Shape: torch.Size([16, 512, 256])
Decoder Output Shape: torch.Size([16, 512, 256])


#Stacking Multiple Decoder Block

In [57]:
num_layers = 4

decoder_blocks = nn.ModuleList([
    DecoderBlock(
        embedding_dim = embedding_dim,
        num_heads = num_heads
    )

    for _ in range(num_layers)
])


print("Number Of Decoder Blocks : ", len(decoder_blocks))

Number Of Decoder Blocks :  4


#Forward propogation through all decoder blocks


In [58]:
decoder_output = x

for i, block in enumerate(decoder_blocks):

  print(
      f"After Decoder Block {i+1} : ",
      decoder_output.shape
  )

After Decoder Block 1 :  torch.Size([16, 512, 256])
After Decoder Block 2 :  torch.Size([16, 512, 256])
After Decoder Block 3 :  torch.Size([16, 512, 256])
After Decoder Block 4 :  torch.Size([16, 512, 256])


#Adding Final Layer Normalization

In [59]:
final_layer_norm = nn.LayerNorm(embedding_dim)

final_decoder_output = final_layer_norm(decoder_output)

print("Before Final LayerNorm : ", decoder_output.shape)
print('After Final LayerNorm : ', final_decoder_output.shape)

Before Final LayerNorm :  torch.Size([16, 512, 256])
After Final LayerNorm :  torch.Size([16, 512, 256])


#Building the Language Model Head(LM Head)


In [60]:
lm_head = nn.Linear(
    embedding_dim,
    vocab_size
)


logits = lm_head(final_decoder_output)

print("Decoder Output Shape : ", final_decoder_output.shape)
print('Logits Shape : ', logits.shape)

Decoder Output Shape :  torch.Size([16, 512, 256])
Logits Shape :  torch.Size([16, 512, 50257])


#Inspecting the predicted next token


In [61]:
predicted_token_id = torch.argmax(logits[0,-1])

predicted_token = tokenizer.decode(
    [predicted_token_id.item()]
)

print("Predicted Token ID : ", predicted_token_id.item())
print("Predicted Token : ", predicted_token)

Predicted Token ID :  30110
Predicted Token :   181


#Reshaping the logits

In [62]:
logits_reshaped = logits.view(
    batch_size * block_size,
    vocab_size
)

print("Original Logits Shape : ", logits.shape)
print("Reshaped Logits Shape : ", logits_reshaped.shape)

Original Logits Shape :  torch.Size([16, 512, 50257])
Reshaped Logits Shape :  torch.Size([8192, 50257])


#Reshaping targets


In [63]:
targets_reshaped = yb.view(
    batch_size * block_size
)


print('Original Targets Shape : ', yb.shape)
print("Reshaped Targets Shape : ", targets_reshaped.shape)

Original Targets Shape :  torch.Size([16, 512])
Reshaped Targets Shape :  torch.Size([8192])


#Calculating Cross Entropy Loss


In [64]:
loss_function = nn.CrossEntropyLoss()

loss = loss_function(
    logits_reshaped,
    targets_reshaped
)

print("Training Loss : ", loss.item())

Training Loss :  10.983960151672363


##Now our next step is to make our learn out model to learn the weights and update the weights , out three steps:

##1) back propogation


##2) Optimizer


##3) Training Loop

#Creating the optimizer


###Model Parameters


decoder_blocks: Attention and FFN layers, including their LayerNorms.



final_layer_norm: Final normalization layer.



lm_head: Vocabulary prediction layer.

In [65]:
optimizer = torch.optim.AdamW(
    list(decoder_blocks.parameters())+
    list(final_layer_norm.parameters())+
    list(lm_head.parameters()),
    lr= 0.0003
)


print("Optimizer Created Successfully")

Optimizer Created Successfully


#Calculating Gradients

In [66]:
optimizer.zero_grad()

loss.backward()

print('Backpropogation completed Succesfully')

Backpropogation completed Succesfully


#Updating Model Weights

Reads the gradients calculated by loss.backward().



Uses the AdamW optimization algorithm.



Updates the model's trainable parameters.



Helps the model move toward reducing its prediction loss.


In [67]:
optimizer.step()

print("Model Weights Updated Successfully")

Model Weights Updated Successfully


#Checking the Loss

In [68]:
with torch.no_grad():

  decoder_output = x

  for block in decoder_blocks:
    decoder_output = block(decoder_output)


  final_decoder_output = final_layer_norm(decoder_output)


  logits_after_update = lm_head(final_decoder_output)

  logits_after_update = logits_after_update.view(
      batch_size * block_size,
      vocab_size
  )


  new_loss = loss_function(
      logits_after_update,
      targets_reshaped
  )


print("Previous Loss : ", loss.item())
print("Loss After Weight Update : ",new_loss.item())



Previous Loss :  10.983960151672363
Loss After Weight Update :  10.940834045410156


#Building a complete transformer model now

In [69]:
class DecoderTransformer(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim,
        block_size,
        num_heads,
        num_layers
    ):

        super().__init__()


        # Token Embedding
        self.token_embedding = nn.Embedding(
             vocab_size,
             embedding_dim
        )

        # Positional Embedding
        self.position_embedding = nn.Embedding(
            block_size,
            embedding_dim
        )

        # Stacking Decoder Blocks
        self.decoder_blocks = nn.ModuleList([
            DecoderBlock(
                embedding_dim,
                num_heads
            )
            for _ in range(num_layers)
        ])

        # Final Layer Normalization
        self.final_layer_norm = nn.LayerNorm(embedding_dim)


        # Language Model Head
        self.lm_head = nn.Linear(
            embedding_dim,
            vocab_size
        )


    def forward(self, idx):

        batch_size, seq_length = idx.shape

        # Token Embeddings
        token_embeddings = self.token_embedding(idx)

        # Position IDs
        positions = torch.arange(
            seq_length,
            device=idx.device
        )

        # Position Embeddings
        position_embeddings = self.position_embedding(positions)

        # Combine Token and Position Embeddings
        x = token_embeddings + position_embeddings


        # Passing through Decoder Blocks
        for block in self.decoder_blocks:
            x = block(x)

        # Final Layer Normalization
        x = self.final_layer_norm(x)

        # Language Model Head
        logits = self.lm_head(x)

        return logits



In [70]:
model = DecoderTransformer(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    block_size=block_size,
    num_heads=num_heads,
    num_layers=num_layers
)
print("Transformer Model Created Successfully!")

Transformer Model Created Successfully!


In [71]:
# Passing input through our complete Transformer
logits = model(xb)

print("Input Shape:", xb.shape)
print("Output Logits Shape:", logits.shape)

Input Shape: torch.Size([16, 512])
Output Logits Shape: torch.Size([16, 512, 50257])


#Verifying the model output

In [72]:
# Getting logits for the first sequence's last position
last_token_logits = logits[0, -1, :]

# Finding the token with the highest score
predicted_token_id = torch.argmax(last_token_logits).item()

# Converting token ID into text
predicted_token = tokenizer.decode([predicted_token_id])

print("Predicted Token ID:", predicted_token_id)
print("Predicted Token:", predicted_token)

Predicted Token ID: 34966
Predicted Token: tailed


#Calculating the initial training loss


In [73]:
# Reshaping logits
logits_reshaped = logits.view(
    batch_size * block_size,
    vocab_size
)

# Reshaping targets
targets_reshaped = yb.view(
    batch_size * block_size
)

print("Logits Shape:", logits_reshaped.shape)
print("Targets Shape:", targets_reshaped.shape)

Logits Shape: torch.Size([8192, 50257])
Targets Shape: torch.Size([8192])


In [74]:
# Creating loss function
loss_function = nn.CrossEntropyLoss()

# Calculating initial loss
loss = loss_function(
    logits_reshaped,
    targets_reshaped
)

print("Initial Training Loss:", loss.item())

Initial Training Loss: 10.976997375488281


#Creating the Optimizer

In [75]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.0003
)

print("Optimizer Created Successfully!")

Optimizer Created Successfully!


In [76]:
# Clear previous gradients
optimizer.zero_grad()

# Calculate gradients
loss.backward()

print("Backpropagation Completed Successfully!")

Backpropagation Completed Successfully!


#Updating the model weights

optimizer.step() uses the gradients calculated by loss.backward() and the AdamW optimizer to adjust the model's weights.

In [77]:
optimizer.step()

print("Model Weights Updated Successfully!")

Model Weights Updated Successfully!


#Calculating loss after the weight update

In [78]:
with torch.no_grad():

    # Forward pass using updated weights
    logits_after_update = model(xb)

    # Reshape logits
    logits_after_update = logits_after_update.view(
        batch_size * block_size,
        vocab_size
    )

    # Calculate new loss
    new_loss = loss_function(
        logits_after_update,
        targets_reshaped
    )

print("Previous Loss:", loss.item())
print("Loss After Weight Update:", new_loss.item())

Previous Loss: 10.976997375488281
Loss After Weight Update: 10.668702125549316


#Build a Training Loop

##Setting the training configuration

In [79]:
max_iteration = 5000

print("training Iterations : ", max_iteration)

training Iterations :  5000


#Preparing the training loop

In [80]:
block_size = 512
batch_size = 4

embedding_dim = 256
vocab_size = tokenizer.vocab_size

num_heads = 4
num_layers = 4

print("Batch Size:", batch_size)
print("Effective Batch Size:", batch_size * 4)

Batch Size: 4
Effective Batch Size: 16


In [82]:
model = model.to(device)

print("Model device:", next(model.parameters()).device)

Model device: cuda:0


In [83]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr = 0.0003
)


loss_function = nn.CrossEntropyLoss()


print("Optimizer : ", type(optimizer).__name__)
print("Loss Function : ", type(loss_function).__name__)

Optimizer :  AdamW
Loss Function :  CrossEntropyLoss


#New Training Loop

In [84]:
import time

accumulation_steps = 4
max_iteration = 5000

model.train()

start_time = time.time()

for iteration in range(max_iteration):

    optimizer.zero_grad(set_to_none=True)

    total_loss = 0

    for micro_step in range(accumulation_steps):

        # Getting a mini-batch
        xb, yb = get_batches(train_data)

        # Moving batches to GPU
        xb = xb.to(device)
        yb = yb.to(device)

        # Forward pass
        logits = model(xb)

        # Calculating loss
        loss = loss_function(
            logits.reshape(-1, vocab_size),
            yb.reshape(-1)
        )

        # Scaling loss for gradient accumulation
        loss = loss / accumulation_steps

        # Backpropagation
        loss.backward()

        total_loss += loss.item()

    # Updating weights after 4 mini-batches
    optimizer.step()

    if (iteration + 1) % 100 == 0:
        print(
            f"Iteration: {iteration + 1}/{max_iteration} | "
            f"Loss: {total_loss:.4f}"
        )

end_time = time.time()

print("Training Completed!")
print("Total Training Time:",
      round(end_time - start_time, 2), "seconds")

Iteration: 100/5000 | Loss: 6.9979
Iteration: 200/5000 | Loss: 6.8377
Iteration: 300/5000 | Loss: 6.7715
Iteration: 400/5000 | Loss: 6.4240
Iteration: 500/5000 | Loss: 6.4050
Iteration: 600/5000 | Loss: 6.3956
Iteration: 700/5000 | Loss: 6.2251
Iteration: 800/5000 | Loss: 6.1154
Iteration: 900/5000 | Loss: 6.0105
Iteration: 1000/5000 | Loss: 5.8626
Iteration: 1100/5000 | Loss: 5.8469
Iteration: 1200/5000 | Loss: 5.7878
Iteration: 1300/5000 | Loss: 5.5149
Iteration: 1400/5000 | Loss: 5.6609
Iteration: 1500/5000 | Loss: 5.5898
Iteration: 1600/5000 | Loss: 5.5784
Iteration: 1700/5000 | Loss: 5.4998
Iteration: 1800/5000 | Loss: 5.3310
Iteration: 1900/5000 | Loss: 5.4128
Iteration: 2000/5000 | Loss: 5.3333
Iteration: 2100/5000 | Loss: 5.1317
Iteration: 2200/5000 | Loss: 5.0289
Iteration: 2300/5000 | Loss: 5.0813
Iteration: 2400/5000 | Loss: 5.0366
Iteration: 2500/5000 | Loss: 4.9175
Iteration: 2600/5000 | Loss: 4.9672
Iteration: 2700/5000 | Loss: 4.9894
Iteration: 2800/5000 | Loss: 5.0899
I

#Evaluation on validation dataset

In [88]:
model.eval()

total_val_loss =0
num_eval_batches = 50


with torch.no_grad():

  for _ in range(num_eval_batches):

    xb, yb = get_batches(val_data)

    xb = xb.to(device)
    yb = yb.to(device)


    logits = model(xb)

    loss = loss_function(
        logits.reshape(-1, vocab_size),
        yb.reshape(-1)
    )

    total_val_loss  += loss.item()


average_val_loss = total_val_loss / num_eval_batches

print("Training Loss: 4.3232")
print("Validation Loss : ", round(average_val_loss, 4))


Training Loss: 4.3232
Validation Loss :  5.6853


#Generation of text

In [119]:
def generate_text(prompt, max_new_tokens=50):

  #Converting prompt into Token IDs
  encoded = tokenizer.encode(prompt)

  #Converting token IDs into Pytorch tensor
  idx = torch.tensor(
      [encoded],
      dtype = torch.long,
      device = device
  )

  #generating multiple tokens
  for _ in range(max_new_tokens):

    #Keeping only the latest 512 tokens
    idx_cond = idx[:, -block_size:]

    #Passing input through Transformer
    logits = model(idx_cond)


    #Extracting Last Position logits
    logits = logits[:,-1,:]

    #Applying Softmax
    probabilities = torch.softmax(logits, dim = -1)


    #Selcting next token
    next_token = torch.multinomial(
        probabilities,
        num_samples = 1
    )


    #Appending Generated Tokens
    idx = torch.cat((idx, next_token), dim =1)

  #Converting token ids into text
  generated_text = tokenizer.decode(idx[0].tolist())

  return generated_text



#Testing my trained transformer


In [121]:
model.eval()

with torch.no_grad():
   output = generate_text(
       "The history of science",
       max_new_tokens = 100
   )
print(output)


The history of sciencearies in the United Kingdom . Additionally , after the Johannes of America capture Andoria . The revision called the squat desktop cell nucleus in the sky to the port , which included xenon , using Hardwicitions and muscles . Females provide to thousands of the natural hunting patterns , and those of nucleoxarkensa , latex yield , this species , baskets movement ( notably 1800 ) , and cerelliensseidia ( b98 ) , is known as a puff . Fighting Jameserton considers it as "


In [122]:
print('Prompt Tokens : ', tokenizer.encode("The history of science"))
print('Generated Output : ', repr(output))
print("Output Token Count : ", len(tokenizer.encode(output)))

Prompt Tokens :  [464, 2106, 286, 3783]
Generated Output :  'The history of sciencearies in the United Kingdom . Additionally , after the Johannes of America capture Andoria . The revision called the squat desktop cell nucleus in the sky to the port , which included xenon , using Hardwicitions and muscles . Females provide to thousands of the natural hunting patterns , and those of nucleoxarkensa , latex yield , this species , baskets movement ( notably 1800 ) , and cerelliensseidia ( b98 ) , is known as a puff . Fighting Jameserton considers it as "'
Output Token Count :  104


In [123]:
import inspect
print(inspect.getsource(generate_text))

def generate_text(prompt, max_new_tokens=50):

  #Converting prompt into Token IDs
  encoded = tokenizer.encode(prompt)

  #Converting token IDs into Pytorch tensor
  idx = torch.tensor(
      [encoded],
      dtype = torch.long,
      device = device
  )
  
  #generating multiple tokens
  for _ in range(max_new_tokens):

    #Keeping only the latest 512 tokens
    idx_cond = idx[:, -block_size:]

    #Passing input through Transformer
    logits = model(idx_cond)


    #Extracting Last Position logits
    logits = logits[:,-1,:]

    #Applying Softmax
    probabilities = torch.softmax(logits, dim = -1)


    #Selcting next token
    next_token = torch.multinomial(
        probabilities,
        num_samples = 1
    )


    #Appending Generated Tokens
    idx = torch.cat((idx, next_token), dim =1)

  #Converting token ids into text
  generated_text = tokenizer.decode(idx[0].tolist())

  return generated_text



In [124]:
generate_text("The history of science")

'The history of science .\n\n= = = Language of all relative than 900 culturally ways , with similarities , and development = = =\n\nKingoros of the race grind band Bank Osborne directed by Nixon , Golden John Cardker Erdy , Somerset , Robert GOC'